# Movie Recommendation Systems | Assignment 8 — Classroom Dataset Verification

**Juan Luna Santibanez**

Miami Dade College | CAI2100C-2267-6908 | Machine Learning Foundations

**Professor Beth Muturi**

**Goal:** Independently verify content-based and user-user collaborative filtering with the authentic classroom MovieLens/TMDB files.

**Important:** This companion notebook uses the original CSVs from the class ZIP. The separate Assignment 8 tasks require IMDb Extensive on Kaggle; the class files must not be described as that dataset.

## Upload the original ZIP

Upload **Assignment 8-machinelearning.zip** from the classroom materials to your notebook environment, or place the ZIP in the working directory. The code below reads CSVs directly from the ZIP without redistributing them.

In [ ]:
import zipfile, pathlib, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
files=list(pathlib.Path('.').rglob('Assignment 8-machinelearning.zip'))
if not files: raise FileNotFoundError('Upload Assignment 8-machinelearning.zip from the course materials.')
z=zipfile.ZipFile(files[0]); prefix='Assignment 8/Assignment 8 Dependencies/'
with z.open(prefix+'movies_metadata.csv') as f: movies=pd.read_csv(f,low_memory=False)
with z.open(prefix+'ratings_small.csv') as f: ratings=pd.read_csv(f)
with z.open(prefix+'links_small.csv') as f: links=pd.read_csv(f)
print('Movie metadata:',len(movies),'ratings:',len(ratings),'ID links:',len(links))
print('Users:',ratings.userId.nunique(),'movies rated:',ratings.movieId.nunique())

## Task 1 — Content similarity with real class metadata

Compare movie overviews with TF-IDF and cosine similarity. The final IMDb notebook uses different columns because the datasets are not identical.

In [ ]:
m=movies[['imdb_id','title','overview','genres','vote_count']].dropna(subset=['title']).copy()
m['overview']=m.overview.fillna('')
m=m[m.overview.str.len()>35].drop_duplicates('imdb_id').head(4000).reset_index(drop=True)
v=TfidfVectorizer(stop_words='english',max_features=4000,ngram_range=(1,2))
x=v.fit_transform(m.overview)
i=int(np.flatnonzero(m.title.str.casefold().eq('toy story'))[0])
sim=linear_kernel(x[i],x).ravel();order=[j for j in np.argsort(-sim) if j!=i][:5]
content_results=pd.DataFrame({'similar_title':m.iloc[order].title.to_numpy(),'cosine_similarity':sim[order]})
display(content_results)

## Task 2 — Collaborative filtering on authentic user ratings

MovieLens `ratings_small.csv` contains `userId`, `movieId`, and `rating` columns. I compare user rating patterns after centering by each user's mean, keep up to 25 positively similar neighbors, and validate held-out ratings.

In [ ]:
train,test=train_test_split(ratings[['userId','movieId','rating']],test_size=.2,random_state=42)
users=np.sort(train.userId.unique());items=np.sort(train.movieId.unique())
uid={k:i for i,k in enumerate(users)};mid={k:i for i,k in enumerate(items)}
row=train.userId.map(uid).to_numpy();col=train.movieId.map(mid).to_numpy();value=train.rating.to_numpy()
mat=csr_matrix((value,(row,col)),shape=(len(users),len(items)))
global_avg=float(train.rating.mean());user_means=train.groupby('userId').rating.mean()
means=np.array([user_means.get(k,global_avg) for k in users])
centered=csr_matrix((value-means[row],(row,col)),shape=mat.shape)
sims=cosine_similarity(centered);np.fill_diagonal(sims,0.)
predictions=[];actual=[]
for user,item,truth in test.itertuples(index=False):
    if user not in uid or item not in mid:continue
    i=uid[user];j=mid[item];votes=mat.getcol(j).toarray().ravel()
    neighbors=np.where((votes>0)&(sims[i]>0))[0]
    if len(neighbors):
        nearest=neighbors[np.argsort(-sims[i,neighbors])[:25]]
        weights=sims[i,nearest]
        estimate=means[i]+np.dot(weights,votes[nearest]-means[nearest])/weights.sum()
    else:estimate=means[i]
    predictions.append(np.clip(estimate,.5,5.));actual.append(truth)
errors=np.array(predictions)-np.array(actual)
print('Held-out ratings with known training IDs:',len(actual))
print('User-user collaborative MAE:',round(np.abs(errors).mean(),4))
print('User-user collaborative RMSE:',round(np.sqrt((errors**2).mean()),4))

## Conclusion

I verified both recommendation approaches using the exact authentic files from class. The real user records in MovieLens allow individual-user collaborative filtering. The final IMDb Extensive dataset provides demographic cohort averages instead; those results should not be confused with the class verification above.